# Прогнозирование энергопотребления домохозяйства методами машинного обучения

**Дисциплина:** «Большие данные и машинное обучение»  
**Тип задачи:** регрессия и прогнозирование временного ряда  
**Данные:** UCI Individual Household Electric Power Consumption  
**Основные инструменты:** Python, Pandas, NumPy, Matplotlib, Scikit-learn, XGBoost

## Цель работы

Построить и сравнить несколько моделей машинного обучения для прогнозирования электрической нагрузки домохозяйства по историческим измерениям. В качестве целевой переменной используется средняя активная мощность за следующий час (`Global_active_power`, кВт).

Исходные измерения записаны с минутным шагом. Для удобства анализа они агрегируются в часовой временной ряд. Средняя мощность за один час в кВт численно равна энергии, потреблённой за этот час, в кВт·ч.

## План исследования

1. Загрузить исходный набор UCI и проверить его структуру и качество.
2. Обработать пропуски и агрегировать минутные данные до часовых.
3. Выполнить первичный анализ и визуализацию временного ряда.
4. Сформировать календарные, лаговые и скользящие признаки.
5. Разделить данные **по времени** на обучающую, валидационную и тестовую выборки.
6. Сравнить на валидации базовые прогнозы и модели Ridge, Random Forest, HistGradientBoosting и XGBoost.
7. Переобучить модели на обучающей и валидационной частях и оценить качество на отложенном тестовом периоде.
8. Построить графики фактических и прогнозных значений, проанализировать ошибки и сформировать прогноз на следующие 24 часа.

> **Важно:** задача реализована как скользящий одношаговый прогноз (rolling one-step-ahead): при прогнозе очередного часа используются фактические измерения, уже поступившие к предыдущему часу. Для отдельного прогноза на 24 часа вперёд ниже дополнительно реализован рекурсивный режим, в котором собственные прогнозы модели становятся входными значениями для следующих шагов.

## 1. Подготовка окружения

В Google Colab большинство библиотек уже установлено. В следующей ячейке устанавливается XGBoost, если он отсутствует или требует обновления. При запуске ноутбука сверху вниз установка выполняется автоматически.

In [ ]:
%pip -q install -U xgboost

In [ ]:
import io
import zipfile
import shutil
import urllib.request
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import xgboost
import joblib
import scipy
import platform

from IPython.display import display
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score,
)
from xgboost import XGBRegressor

SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

## 2. Загрузка набора данных

Набор данных опубликован в UCI Machine Learning Repository. Он содержит 2 075 259 минутных наблюдений одного домохозяйства за период с декабря 2006 года по ноябрь 2010 года. В исходном файле девять столбцов: дата, время, активная и реактивная мощность, напряжение, сила тока и три показателя субучёта.

Источник и описание: [UCI — Individual Household Electric Power Consumption](https://archive.ics.uci.edu/dataset/235/individual+household+electric+power+consumption). Архив загружается напрямую с официального сервера UCI.

In [ ]:
DATA_DIR = Path("/content/household_power_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
RAW_FILE = DATA_DIR / "household_power_consumption.txt"
ZIP_FILE = DATA_DIR / "household_power_consumption.zip"

UCI_ZIP_URL = (
    "https://archive.ics.uci.edu/static/public/235/"
    "individual+household+electric+power+consumption.zip"
)

if not RAW_FILE.exists():
    print("Загружаю архив UCI...")
    urllib.request.urlretrieve(UCI_ZIP_URL, ZIP_FILE)

    print("Распаковываю архив...")
    with zipfile.ZipFile(ZIP_FILE, "r") as archive:
        member = next(
            name for name in archive.namelist()
            if name.lower().endswith("household_power_consumption.txt")
        )
        with archive.open(member) as source, open(RAW_FILE, "wb") as destination:
            shutil.copyfileobj(source, destination)
    print("Файл подготовлен:", RAW_FILE)
else:
    print("Исходный файл уже существует, повторная загрузка не требуется.")

print(f"Размер исходного файла: {RAW_FILE.stat().st_size / (1024**2):.1f} МБ")

In [ ]:
# В исходном файле разделитель столбцов — точка с запятой,
# а пропущенные значения обозначены пустыми полями.
raw = pd.read_csv(
    RAW_FILE,
    sep=";",
    na_values=["?", ""],
    low_memory=False,
)

# Приводим измерительные столбцы к числовому формату.
measurement_columns = [column for column in raw.columns if column not in ["Date", "Time"]]
for column in measurement_columns:
    raw[column] = pd.to_numeric(raw[column], errors="coerce")

# Собираем дату и время в единый индекс.
raw["datetime"] = pd.to_datetime(
    raw["Date"] + " " + raw["Time"],
    format="%d/%m/%Y %H:%M:%S",
    errors="coerce",
)
raw = raw.dropna(subset=["datetime"]).sort_values("datetime")
raw = raw.set_index("datetime")

print("Размер таблицы:", raw.shape)
print("Период:", raw.index.min(), "—", raw.index.max())
print("Число повторяющихся временных меток:", raw.index.duplicated().sum())
display(raw.head())

In [ ]:
# Проверяем пропуски до их обработки.
missing_summary = raw[measurement_columns].isna().sum().sort_values(ascending=False)
missing_percent = (missing_summary / len(raw) * 100).round(3)
quality_table = pd.DataFrame({
    "Пропусков": missing_summary,
    "Доля пропусков, %": missing_percent,
})
display(quality_table)

print(f"Всего строк в исходных данных: {len(raw):,}")
print(f"Доля строк с пропуском хотя бы в одном измерении: {raw[measurement_columns].isna().any(axis=1).mean() * 100:.3f}%")

## 3. Очистка и переход к часовому временному ряду

Целевая переменная — `Global_active_power`, то есть средняя активная мощность домохозяйства в кВт на минутном интервале.

Пропуски в целевой переменной обрабатываются временной интерполяцией, но только в пределах не более 60 последовательных минут и только внутри временного диапазона. Дополнительно для каждого часа проверяется полнота исходных наблюдений: часы, в которых до интерполяции было менее 45 корректных минутных значений, исключаются из целевого ряда. После этого создаётся регулярная часовая сетка. Если отдельный час остаётся без значения, он сохранится как пропуск; при формировании лагов строки с неполными окнами будут исключены.

Такой подход позволяет не заполнять большие разрывы произвольными значениями и не скрывать низкое качество отдельных интервалов.

In [ ]:
TARGET_RAW = "Global_active_power"

# Число реально наблюдавшихся (не пропущенных) минутных значений в каждом часу.
valid_minutes_per_hour = raw[TARGET_RAW].notna().astype("int8").resample("h").sum()

# Временная интерполяция только небольших внутренних пропусков.
target_minute = raw[TARGET_RAW].interpolate(
    method="time",
    limit=60,
    limit_area="inside",
)

# Средняя активная мощность за каждый час.
hourly_target = target_minute.resample("h").mean()

# Не используем часы, в которых исходных измерений было меньше 45 минут.
valid_minutes_per_hour = valid_minutes_per_hour.reindex(hourly_target.index).fillna(0)
hourly_target.loc[valid_minutes_per_hour < 45] = np.nan

hourly = hourly_target.to_frame(name="target_kw").asfreq("h")

print("Число часовых интервалов:", len(hourly))
print("Часов без значения целевой переменной:", int(hourly["target_kw"].isna().sum()))
print("Начало ряда:", hourly.index.min())
print("Конец ряда:", hourly.index.max())
display(hourly.head())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 9))

hourly["target_kw"].plot(ax=axes[0], linewidth=0.6)
axes[0].set_title("Средняя активная мощность домохозяйства: часовой ряд")
axes[0].set_ylabel("Мощность, кВт")
axes[0].set_xlabel("Дата")
axes[0].grid(True, alpha=0.3)

# Фрагмент временного ряда для более подробного рассмотрения.
start_example = hourly["target_kw"].first_valid_index()
if start_example is not None:
    hourly.loc[start_example : start_example + pd.Timedelta(days=7), "target_kw"].plot(
        ax=axes[1], linewidth=1.0
    )
axes[1].set_title("Пример динамики энергопотребления за первую доступную неделю")
axes[1].set_ylabel("Мощность, кВт")
axes[1].set_xlabel("Дата")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Исследовательский анализ данных

Рассмотрим распределение целевой переменной и средние значения мощности в зависимости от часа суток и дня недели. Эти зависимости обосновывают включение календарных признаков в модели.

In [ ]:
valid_power = hourly["target_kw"].dropna()
print("Описательная статистика часовой средней мощности (кВт):")
display(valid_power.describe().to_frame("Значение"))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
valid_power.hist(bins=50, ax=axes[0])
axes[0].set_title("Распределение часовой средней мощности")
axes[0].set_xlabel("Мощность, кВт")
axes[0].set_ylabel("Частота")
axes[0].grid(True, alpha=0.25)

by_hour = valid_power.groupby(valid_power.index.hour).mean()
by_hour.plot(kind="bar", ax=axes[1])
axes[1].set_title("Средняя мощность в зависимости от часа суток")
axes[1].set_xlabel("Час суток")
axes[1].set_ylabel("Средняя мощность, кВт")
axes[1].grid(True, axis="y", alpha=0.25)

plt.tight_layout()
plt.show()

In [ ]:
weekday_names = ["Пн", "Вт", "Ср", "Чт", "Пт", "Сб", "Вс"]
by_weekday = valid_power.groupby(valid_power.index.dayofweek).mean()
by_weekday.index = [weekday_names[i] for i in by_weekday.index]

plt.figure(figsize=(9, 4))
by_weekday.plot(kind="bar")
plt.title("Средняя мощность в зависимости от дня недели")
plt.xlabel("День недели")
plt.ylabel("Средняя мощность, кВт")
plt.grid(True, axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## 5. Формирование признаков для машинного обучения

Модели получают информацию, которая действительно доступна к моменту прогноза:

- **Лаговые признаки:** мощность за предыдущие 1, 2, 3, 6, 12, 24, 48, 72, 168 и 336 часов.
- **Скользящие статистики:** среднее и стандартное отклонение за предыдущие 3, 6, 12, 24 и 168 часов.
- **Календарные признаки:** час суток, день недели и месяц в циклическом представлении, а также признак выходного дня.

При расчёте скользящих статистик применяется сдвиг на один час (`shift(1)`). Следовательно, признаки для часа `t` не содержат целевое значение этого же часа. Исходные субсчётчики и другие измерения текущего часа не используются: во время реального прогноза на следующий час они ещё неизвестны.

In [ ]:
LAGS = [1, 2, 3, 6, 12, 24, 48, 72, 168, 336]
ROLLING_WINDOWS = [3, 6, 12, 24, 168]


def make_calendar_features(index):
    """Календарные признаки для переданного DatetimeIndex."""
    index = pd.DatetimeIndex(index)
    result = pd.DataFrame(index=index)

    hour = index.hour
    dayofweek = index.dayofweek
    month = index.month
    dayofyear = index.dayofyear

    result["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    result["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    result["weekday_sin"] = np.sin(2 * np.pi * dayofweek / 7)
    result["weekday_cos"] = np.cos(2 * np.pi * dayofweek / 7)
    result["month_sin"] = np.sin(2 * np.pi * (month - 1) / 12)
    result["month_cos"] = np.cos(2 * np.pi * (month - 1) / 12)
    result["dayofyear_sin"] = np.sin(2 * np.pi * (dayofyear - 1) / 365.25)
    result["dayofyear_cos"] = np.cos(2 * np.pi * (dayofyear - 1) / 365.25)
    result["is_weekend"] = (dayofweek >= 5).astype("int8")
    return result


def make_features(target_series):
    """Создаёт признаки строго из прошлых значений и календаря."""
    features = pd.DataFrame(index=target_series.index)

    for lag in LAGS:
        features[f"lag_{lag}"] = target_series.shift(lag)

    past_values = target_series.shift(1)
    for window in ROLLING_WINDOWS:
        features[f"rolling_mean_{window}"] = past_values.rolling(window=window).mean()
        features[f"rolling_std_{window}"] = past_values.rolling(window=window).std()

    features = features.join(make_calendar_features(features.index))
    return features


X_all = make_features(hourly["target_kw"])
y_all = hourly["target_kw"].rename("target_kw")

# Удаляем строки, для которых целевая переменная или исторические признаки недоступны.
prepared = X_all.join(y_all).dropna()
feature_columns = X_all.columns.tolist()
X = prepared[feature_columns]
y = prepared["target_kw"]

print("Число исходных часовых интервалов:", len(hourly))
print("Число строк после формирования признаков:", len(prepared))
print("Число признаков:", len(feature_columns))
print("Признаки:", feature_columns)
display(prepared.head())

## 6. Хронологическое разделение выборки

Данные делятся без случайного перемешивания:

- 70% — обучение;
- 15% — валидация и выбор лучшей модели;
- 15% — финальное тестирование.

Тестовый период не используется ни для обучения, ни для выбора модели. Такой порядок особенно важен для временных рядов: случайное перемешивание могло бы передать модели информацию из будущего.

In [ ]:
n_rows = len(prepared)
train_end = int(n_rows * 0.70)
val_end = int(n_rows * 0.85)

train = prepared.iloc[:train_end]
validation = prepared.iloc[train_end:val_end]
test = prepared.iloc[val_end:]

X_train, y_train = train[feature_columns], train["target_kw"]
X_val, y_val = validation[feature_columns], validation["target_kw"]
X_test, y_test = test[feature_columns], test["target_kw"]

print("Обучающая выборка:", len(train), train.index.min(), "—", train.index.max())
print("Валидационная выборка:", len(validation), validation.index.min(), "—", validation.index.max())
print("Тестовая выборка:", len(test), test.index.min(), "—", test.index.max())

## 7. Метрики качества

Для сравнения используются следующие метрики:

- **MAE** — средняя абсолютная ошибка, кВт. Чем меньше, тем лучше.
- **RMSE** — корень из средней квадратичной ошибки, кВт. Сильнее штрафует большие отклонения.
- **MAPE** — средняя абсолютная процентная ошибка. Чтобы избежать неустойчивости при очень малых значениях фактической мощности, MAPE рассчитывается только для часов, где фактическая мощность не ниже 0,1 кВт.
- **WAPE** — суммарная абсолютная ошибка относительно суммарного фактического потребления, %.
- **R²** — коэффициент детерминации. Чем ближе к 1, тем большую долю изменчивости ряда объясняет модель.

Для выбора модели основным критерием будет RMSE на валидационной выборке. Все метрики интерпретируются совместно.

In [ ]:
def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    absolute_error = np.abs(y_true - y_pred)

    mape_mask = y_true >= 0.1
    if mape_mask.any():
        mape = mean_absolute_percentage_error(y_true[mape_mask], y_pred[mape_mask]) * 100
    else:
        mape = np.nan

    wape = absolute_error.sum() / np.abs(y_true).sum() * 100 if np.abs(y_true).sum() else np.nan

    return {
        "MAE, кВт": mean_absolute_error(y_true, y_pred),
        "RMSE, кВт": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAPE, % (y ≥ 0.1 кВт)": mape,
        "WAPE, %": wape,
        "R²": r2_score(y_true, y_pred),
    }

## 8. Базовые прогнозы и модели машинного обучения

Перед обучением сложных алгоритмов проверяются два простых ориентира:

1. **Наивный прогноз:** мощность следующего часа равна мощности предыдущего часа (`lag_1`).
2. **Сезонный наивный прогноз:** мощность следующего часа равна мощности в тот же час предыдущих суток (`lag_24`).

Затем сравниваются четыре регрессионные модели:

- Ridge Regression — линейная модель с L2-регуляризацией;
- Random Forest Regressor — ансамбль деревьев решений;
- HistGradientBoosting Regressor — градиентный бустинг из Scikit-learn;
- XGBoost Regressor — градиентный бустинг библиотеки XGBoost.

Параметры заданы явно для воспроизводимого сравнения. Полный перебор гиперпараметров в рамках этой работы не проводится.

In [ ]:
def build_models():
    """Возвращает новые экземпляры моделей с фиксированными параметрами."""
    return {
        "Ridge": Pipeline([
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0)),
        ]),
        "Random Forest": RandomForestRegressor(
            n_estimators=250,
            max_depth=18,
            min_samples_leaf=2,
            max_features=0.9,
            n_jobs=-1,
            random_state=SEED,
        ),
        "HistGradientBoosting": HistGradientBoostingRegressor(
            learning_rate=0.06,
            max_iter=250,
            max_leaf_nodes=31,
            l2_regularization=1.0,
            early_stopping=False,
            random_state=SEED,
        ),
        "XGBoost": XGBRegressor(
            n_estimators=500,
            max_depth=6,
            learning_rate=0.04,
            subsample=0.8,
            colsample_bytree=0.85,
            reg_lambda=1.0,
            objective="reg:squarederror",
            eval_metric="rmse",
            tree_method="hist",
            n_jobs=-1,
            random_state=SEED,
        ),
    }


models_validation = build_models()
validation_rows = []

# Базовые прогнозы на валидационном отрезке.
validation_rows.append({
    "Модель": "Naive (последний час)",
    **regression_metrics(y_val, X_val["lag_1"]),
})
validation_rows.append({
    "Модель": "Seasonal naive (24 часа)",
    **regression_metrics(y_val, X_val["lag_24"]),
})

# Обучаем модели только на train и оцениваем на validation.
for model_name, model in models_validation.items():
    print(f"Обучение: {model_name}")
    model.fit(X_train, y_train)
    prediction = model.predict(X_val)
    validation_rows.append({
        "Модель": model_name,
        **regression_metrics(y_val, prediction),
    })

validation_results = pd.DataFrame(validation_rows).set_index("Модель").sort_values("RMSE, кВт")
print("Результаты на валидационной выборке:")
display(validation_results)

# Выбираем лучшую ML-модель по RMSE, не включая базовые прогнозы.
best_model_name = min(
    models_validation.keys(),
    key=lambda name: validation_results.loc[name, "RMSE, кВт"],
)
print("Лучшая модель машинного обучения по валидационному RMSE:", best_model_name)

## 9. Финальная оценка на тестовой выборке

После выбора конфигурации каждая модель обучается заново на объединённых обучающей и валидационной выборках. Тестовый отрезок остаётся полностью отложенным и используется только для финальной оценки.

Базовые прогнозы строятся непосредственно по соответствующим лаговым признакам тестовой выборки. Это сопоставление показывает, дают ли алгоритмы машинного обучения преимущество по сравнению с простыми временными эвристиками.

In [ ]:
X_train_val = pd.concat([X_train, X_val]).sort_index()
y_train_val = pd.concat([y_train, y_val]).sort_index()

final_models = {}
test_rows = []

# Базовые модели.
test_rows.append({
    "Модель": "Naive (последний час)",
    **regression_metrics(y_test, X_test["lag_1"]),
})
test_rows.append({
    "Модель": "Seasonal naive (24 часа)",
    **regression_metrics(y_test, X_test["lag_24"]),
})

# Переобучение ML-моделей на train + validation.
for model_name, model in build_models().items():
    print(f"Финальное обучение: {model_name}")
    model.fit(X_train_val, y_train_val)
    final_models[model_name] = model

    prediction = model.predict(X_test)
    test_rows.append({
        "Модель": model_name,
        **regression_metrics(y_test, prediction),
    })

test_results = pd.DataFrame(test_rows).set_index("Модель").sort_values("RMSE, кВт")
print("Итоговое сравнение на тестовой выборке:")
display(test_results)

best_final_model = final_models[best_model_name]
best_test_prediction = best_final_model.predict(X_test)

In [ ]:
# Сравнение RMSE на тестовом периоде.
plt.figure(figsize=(11, 5))
test_results["RMSE, кВт"].sort_values().plot(kind="bar")
plt.title("Сравнение моделей по RMSE на тестовой выборке")
plt.xlabel("Модель")
plt.ylabel("RMSE, кВт (меньше — лучше)")
plt.xticks(rotation=25, ha="right")
plt.grid(True, axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

# Вывод выбранной модели и её тестовых метрик.
print("Модель, выбранная по валидационной выборке:", best_model_name)
print("Её качество на тестовой выборке:")
display(test_results.loc[[best_model_name]])

## 10. Визуальная оценка прогноза

На графике ниже показан тестовый период. Для читаемости выводится последний фрагмент длиной до семи суток (168 часов). Также строится график остатков — разности между фактической и прогнозной мощностью.

In [ ]:
plot_hours = min(24 * 7, len(y_test))
plot_index = y_test.index[-plot_hours:]

plt.figure(figsize=(15, 5))
plt.plot(plot_index, y_test.loc[plot_index], label="Фактическое значение", linewidth=1.2)
plt.plot(
    plot_index,
    pd.Series(best_test_prediction, index=y_test.index).loc[plot_index],
    label=f"Прогноз ({best_model_name})",
    linewidth=1.2,
)
plt.title("Фактическое и прогнозное энергопотребление: тестовый период")
plt.xlabel("Дата")
plt.ylabel("Средняя активная мощность, кВт")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

residuals = y_test.to_numpy() - best_test_prediction

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(residuals, bins=45)
axes[0].set_title("Распределение ошибок прогноза")
axes[0].set_xlabel("Остаток: факт − прогноз, кВт")
axes[0].set_ylabel("Частота")
axes[0].grid(True, alpha=0.25)

axes[1].scatter(y_test, best_test_prediction, alpha=0.25, s=10)
min_value = min(float(y_test.min()), float(best_test_prediction.min()))
max_value = max(float(y_test.max()), float(best_test_prediction.max()))
axes[1].plot([min_value, max_value], [min_value, max_value], linestyle="--")
axes[1].set_title("Фактические и прогнозные значения")
axes[1].set_xlabel("Факт, кВт")
axes[1].set_ylabel("Прогноз, кВт")
axes[1].grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

## 11. Анализ важности признаков

Для моделей, поддерживающих встроенную оценку важности признаков, отображается `feature_importances_`. Для линейной модели Ridge используется модуль коэффициентов после стандартизации признаков. Если выбранная модель не предоставляет такой атрибут, строится перестановочная важность на подвыборке тестовых наблюдений.

Важность признака показывает его вклад в работу конкретной модели и не должна интерпретироваться как доказательство причинно-следственной связи.

In [ ]:
importance = None
importance_label = ""

if hasattr(best_final_model, "feature_importances_"):
    importance = pd.Series(best_final_model.feature_importances_, index=feature_columns)
    importance_label = "Встроенная важность признаков"
elif isinstance(best_final_model, Pipeline) and hasattr(best_final_model.named_steps["model"], "coef_"):
    importance = pd.Series(
        np.abs(best_final_model.named_steps["model"].coef_),
        index=feature_columns,
    )
    importance_label = "Модуль стандартизированных коэффициентов Ridge"
else:
    from sklearn.inspection import permutation_importance
    sample_size = min(1500, len(X_test))
    X_sample = X_test.iloc[:sample_size]
    y_sample = y_test.iloc[:sample_size]
    perm = permutation_importance(
        best_final_model,
        X_sample,
        y_sample,
        n_repeats=3,
        random_state=SEED,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
    )
    importance = pd.Series(perm.importances_mean, index=feature_columns)
    importance_label = "Перестановочная важность (изменение качества при перемешивании признака)"

importance = importance.sort_values(ascending=False).head(15).sort_values()
plt.figure(figsize=(10, 6))
importance.plot(kind="barh")
plt.title(f"{importance_label}: {best_model_name}")
plt.xlabel("Оценка важности")
plt.ylabel("Признак")
plt.grid(True, axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

## 12. Прогноз на следующие 24 часа

После финального сравнения выбранная модель переобучается на всей доступной истории. Затем формируется рекурсивный прогноз на 24 часа вперёд: прогноз первого часа добавляется в историю и используется при расчёте признаков второго часа и так далее.

В отличие от тестирования rolling one-step-ahead, здесь для будущих часов фактические измерения недоступны. Поэтому ошибка может накапливаться по мере увеличения горизонта. Перед расчётом проверяется наличие непрерывной истории, необходимой для лагов и скользящих статистик.

In [ ]:
# Обучаем выбранную модель на всей доступной размеченной истории.
deployment_model = build_models()[best_model_name]
deployment_model.fit(X, y)

# Для рекурсивного прогноза нужны последние 336 последовательных часовых значений.
forecast_history = hourly["target_kw"].asfreq("h").interpolate(
    method="time",
    limit=3,
    limit_direction="both",
)
required_history = max(max(LAGS), max(ROLLING_WINDOWS))
recent_history = forecast_history.tail(required_history)

if len(recent_history) < required_history or recent_history.isna().any():
    print(
        "Нельзя надёжно построить рекурсивный прогноз на 24 часа: "
        "в последнем историческом окне недостаточно непрерывных значений. "
        "Тестовая оценка моделей выше остаётся действительной."
    )
    future_forecast = None
else:
    recursive_history = forecast_history.copy()
    future_index = pd.date_range(
        start=recursive_history.index[-1] + pd.Timedelta(hours=1),
        periods=24,
        freq="h",
    )
    forecast_values = []

    for timestamp in future_index:
        row = {}

        for lag in LAGS:
            row[f"lag_{lag}"] = recursive_history.loc[
                timestamp - pd.Timedelta(hours=lag)
            ]

        for window in ROLLING_WINDOWS:
            previous_window = recursive_history.loc[
                timestamp - pd.Timedelta(hours=window):
                timestamp - pd.Timedelta(hours=1)
            ]
            if len(previous_window) != window or previous_window.isna().any():
                raise ValueError(f"Недостаточно непрерывной истории для окна {window} часов.")
            row[f"rolling_mean_{window}"] = previous_window.mean()
            row[f"rolling_std_{window}"] = previous_window.std()

        row_frame = pd.DataFrame([row], index=pd.DatetimeIndex([timestamp]))
        row_frame = row_frame.join(make_calendar_features(row_frame.index))
        row_frame = row_frame[feature_columns]

        prediction = float(deployment_model.predict(row_frame)[0])
        prediction = max(0.0, prediction)  # мощность не может быть отрицательной
        forecast_values.append(prediction)
        recursive_history.loc[timestamp] = prediction

    future_forecast = pd.Series(
        forecast_values,
        index=future_index,
        name="forecast_kw",
    )

    display(future_forecast.to_frame("Прогноз средней мощности, кВт"))

    plt.figure(figsize=(14, 5))
    history_to_plot = forecast_history.dropna().tail(7 * 24)
    plt.plot(history_to_plot.index, history_to_plot.values, label="Исторические значения")
    plt.plot(future_forecast.index, future_forecast.values, marker="o", label="Прогноз на 24 часа")
    plt.axvline(future_forecast.index[0], linestyle="--", alpha=0.7, label="Начало прогноза")
    plt.title(f"Рекурсивный прогноз на 24 часа: {best_model_name}")
    plt.xlabel("Дата и время")
    plt.ylabel("Средняя активная мощность, кВт")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 13. Сохранение результатов

Тестовые прогнозы и таблицы метрик сохраняются в CSV. При выполнении в Google Colab файлы появятся в рабочей директории `/content/`; при необходимости их можно скачать через интерфейс Colab или выполнить следующую ячейку.

In [ ]:
prediction_export = pd.DataFrame(
    {
        "actual_kw": y_test,
        "predicted_kw": best_test_prediction,
        "error_kw": y_test.to_numpy() - best_test_prediction,
    },
    index=y_test.index,
)
prediction_export.index.name = "datetime"
prediction_export.to_csv("/content/test_predictions.csv", encoding="utf-8-sig")
validation_results.to_csv("/content/validation_metrics.csv", encoding="utf-8-sig")
test_results.to_csv("/content/test_metrics.csv", encoding="utf-8-sig")

if future_forecast is not None:
    future_forecast.to_frame().to_csv("/content/next_24_hours_forecast.csv", encoding="utf-8-sig")

print("Сохранены файлы:")
for filename in ["test_predictions.csv", "validation_metrics.csv", "test_metrics.csv", "next_24_hours_forecast.csv"]:
    path = Path("/content") / filename
    if path.exists():
        print(f"- {path} ({path.stat().st_size:,} байт)")

## 14. Экспорт артефактов для Streamlit-интерфейса

После обучения и оценки моделей эта секция сохраняет всё необходимое для отдельного веб-приложения: выбранную модель, часовой исторический ряд, метрики и тестовые прогнозы. Streamlit не переобучает модель при запуске — он загружает эти файлы и использует их для отображения результатов и построения нового прогноза.

После выполнения ячейки будет создан архив `streamlit_artifacts.zip`. Скачайте его из Colab и распакуйте папку `artifacts` в корень проекта Streamlit, рядом с `app.py`.

> Файл модели Joblib предназначен для использования только с собственными артефактами, созданными этим ноутбуком. Не загружайте в приложение неизвестные `.joblib`/pickle-файлы.


In [ ]:
# Экспортируем артефакты, необходимые Streamlit-приложению.
import json
import joblib
import zipfile
import platform
import sklearn
import xgboost
import scipy

ARTIFACT_ROOT = Path("/content/streamlit_export")
ARTIFACT_DIR = ARTIFACT_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Модель, переобученная на всей доступной подготовленной истории.
joblib.dump(deployment_model, ARTIFACT_DIR / "best_model.joblib", compress=3)

# 2. Исторический часовой ряд.
history_export = hourly[["target_kw"]].copy()
history_export.index.name = "datetime"
history_export.to_csv(ARTIFACT_DIR / "hourly_history.csv", encoding="utf-8-sig")

# 3. Таблицы результатов.
test_predictions_export = pd.DataFrame(
    {
        "actual_kw": y_test,
        "predicted_kw": best_test_prediction,
        "error_kw": y_test.to_numpy() - best_test_prediction,
    },
    index=y_test.index,
)
test_predictions_export.index.name = "datetime"
test_predictions_export.to_csv(
    ARTIFACT_DIR / "test_predictions.csv",
    encoding="utf-8-sig",
)
validation_results.to_csv(ARTIFACT_DIR / "validation_metrics.csv", encoding="utf-8-sig")
test_results.to_csv(ARTIFACT_DIR / "test_metrics.csv", encoding="utf-8-sig")

# 4. Метаданные позволяют приложению восстановить тот же порядок признаков.
metadata = {
    "project_title": "Прогнозирование энергопотребления домохозяйства",
    "model_name": best_model_name,
    "target_column": "target_kw",
    "target_description": "Средняя активная мощность за час, кВт",
    "frequency": "h",
    "forecast_type": "recursive_multi_step",
    "lags": [int(value) for value in LAGS],
    "rolling_windows": [int(value) for value in ROLLING_WINDOWS],
    "feature_columns": [str(value) for value in feature_columns],
    "history_start": str(hourly.index.min()),
    "history_end": str(hourly.index.max()),
    "best_validation_rmse_kw": float(validation_results.loc[best_model_name, "RMSE, кВт"]),
    # Версии фиксируются для воспроизводимой загрузки модели в Streamlit.
    "environment": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "xgboost": xgboost.__version__,
        "scipy": scipy.__version__,
        "joblib": joblib.__version__,
    },
}
with open(ARTIFACT_DIR / "metadata.json", "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

# Отдельный файл со списком ключевых версий для локального окружения.
with open(ARTIFACT_DIR / "model_environment.txt", "w", encoding="utf-8") as file:
    for package_name, package_version in metadata["environment"].items():
        file.write(f"{package_name}=={package_version}\n")

# Архив хранит папку artifacts/ внутри, чтобы её можно было распаковать
# непосредственно в корневую папку Streamlit-проекта.
ARTIFACT_ZIP = Path("/content/streamlit_artifacts.zip")
with zipfile.ZipFile(ARTIFACT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for artifact_path in sorted(ARTIFACT_DIR.iterdir()):
        archive.write(artifact_path, arcname=f"artifacts/{artifact_path.name}")

print("Архив для Streamlit создан:", ARTIFACT_ZIP)
print("Содержимое:")
for artifact_path in sorted(ARTIFACT_DIR.iterdir()):
    print(f"- {artifact_path.name}: {artifact_path.stat().st_size:,} байт")

# В Google Colab сразу предлагаем скачать архив. При запуске вне Colab
# файл останется в /content/streamlit_artifacts.zip.
try:
    from google.colab import files
    files.download(str(ARTIFACT_ZIP))
except Exception:
    print("Автоскачивание доступно в Google Colab. Файл сохранён по указанному пути.")


## 15. Итоговые выводы

После выполнения всех ячеек заполните выводы фактическими результатами из таблиц и графиков. Не следует заранее приписывать конкретной модели преимущество: оно должно подтверждаться метриками на отложенной тестовой выборке.

Рекомендуемая структура вывода:

1. Указать временной диапазон исходного набора и количество записей после очистки и агрегации.
2. Описать выявленные закономерности: изменение нагрузки в течение суток и недели, диапазон значений и наличие пропусков.
3. Назвать модель с наименьшим RMSE на валидации и привести её тестовые MAE, RMSE, MAPE и WAPE.
4. Сравнить выбранную модель с наивным и сезонным наивным прогнозами. Отметить, действительно ли сложные алгоритмы дают выигрыш.
5. Описать характер ошибок по графику остатков и ограничения прогноза на 24 часа.

### Ограничения исследования

- Данные описывают одно домохозяйство во Франции в 2006–2010 годах; результаты нельзя автоматически переносить на другие дома и современные энергосистемы.
- В наборе нет сведений о погоде, тарифах, составе жильцов и режимах работы приборов.
- Используется одношаговый прогноз и рекурсивный 24-часовой прогноз; для более длинных горизонтов может потребоваться отдельная постановка задачи.
- Параметры моделей не подбирались исчерпывающим поиском.
- Интерполяция предназначена для небольших пропусков. Большие разрывы не заполняются автоматически в исходном минутном ряду.

### Источник данных

Hebrail, G., Berard, A. (2006). *Individual Household Electric Power Consumption* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C58K54. Лицензия набора данных: CC BY 4.0.